# Dengue Invaders — Fase 2: consolidación del dataset de mosquitos

Descarga ~19 datasets públicos (Roboflow Universe + Kaggle), normaliza sus etiquetas a una
taxonomía común, deduplica por pHash y produce dos datasets listos para entrenar:

- `detector/` — YOLO, una sola clase `mosquito` (Fase 3)
- `clasificador/` — recortes por especie: `aegypti`, `albopictus`, `culex`, `anopheles`,
  `otro_mosquito`, `no_es_mosquito` (Fase 4)

Plan completo: `docs/PLAN_V5_MODELO_IA.md` del repo *build-with-fable*.

## Antes de correr

1. **Add-ons → Secrets → Add Secret**: label `ROBOFLOW_API_KEY`, valor = tu clave privada de
   Roboflow (Workspace Settings → API Keys). Activá el toggle del secret para ESTE notebook.
2. **Settings → Internet: ON.**
3. **Settings → Persistence: Files only.**
4. No hace falta GPU en esta fase.

La primera celda de código verifica el secret y corta con un mensaje claro si falta.

> ⚠️ Este notebook se genera desde el repo con `ml/notebooks/build_notebook.py`.
> Si editás algo acá, llevá el cambio al repo o se pierde en la próxima regeneración.

In [ ]:
# 1) Verificación del secret — falla rápido, antes de instalar o descargar nada.
from kaggle_secrets import UserSecretsClient

try:
    _clave = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
    assert _clave and len(_clave) > 10
    print("✅ ROBOFLOW_API_KEY encontrada.")
except Exception as e:
    raise SystemExit(
        "❌ Falta el secret ROBOFLOW_API_KEY.\n"
        "   Add-ons → Secrets → Add Secret (label ROBOFLOW_API_KEY) y activalo para este notebook.\n"
        f"   Detalle: {type(e).__name__}"
    )
finally:
    _clave = None  # no dejar la clave viva en el namespace del notebook

In [ ]:
!pip -q install roboflow imagehash pyyaml pillow tqdm kagglehub

## Configuración (copiada del repo)

In [ ]:
!mkdir -p /kaggle/working/config

In [ ]:
%%writefile /kaggle/working/config/taxonomia.yaml
# Taxonomía canónica del modelo de reconocimiento de mosquitos (plan v5, Fase 0).
#
# Cada dataset público etiqueta distinto ("Aedes aegypti", "aegypti", "Ae-aegypti",
# "Aedes aegypti landing", "Aedes_aegypti", "AEDES"...). Este archivo es la única fuente de
# verdad para mapear todo eso a un conjunto fijo de clases.
#
# Lo aplica ml/notebooks/01_consolidar_dataset.py. Al cambiar algo acá hay que regenerar el
# dataset consolidado y anotar la versión nueva (ver `version` abajo).

version: 1

# --- Clases del CLASIFICADOR de especie (Etapa B) --------------------------------------------
# 'otro_mosquito' y 'no_es_mosquito' no son relleno: sin ellas el softmax de 4 clases responde
# una especie objetivo ante CUALQUIER foto (una mosca, un dedo). Ver PLAN_V5 §1.2.
clases_canonicas:
  - aegypti          # Aedes aegypti — vector principal del dengue
  - albopictus       # Aedes albopictus — mosquito tigre
  - culex            # Culex spp.
  - anopheles        # Anopheles spp.
  - otro_mosquito    # culícido real, pero fuera de las 4 objetivo (Culiseta, Armigeres, Ae. vexans...)
  - no_es_mosquito   # cualquier otra cosa: otros insectos, basura, fondo

# --- Etiquetas AMBIGUAS ------------------------------------------------------------------------
# Sirven para entrenar el detector (son mosquitos), pero NO para el clasificador: no se puede
# aprender a separar aegypti de albopictus con imágenes rotuladas solo "AEDES".
# El script las incluye en el dataset del detector y las descarta en el del clasificador.
ambiguas:
  - aedes_sp         # "AEDES" sin especie
  - mosquito_sp      # "mosquito", "objects", genéricos

# --- Detector (Etapa A) ------------------------------------------------------------------------
# Todo lo que sea mosquito colapsa a una sola clase: el detector solo responde "acá hay uno".
detector:
  clase_unica: mosquito
  incluye:
    - aegypti
    - albopictus
    - culex
    - anopheles
    - otro_mosquito
    - aedes_sp
    - mosquito_sp
  excluye:
    - no_es_mosquito

# --- Mapeo ------------------------------------------------------------------------------------
# El script normaliza cada etiqueta antes de comparar: minúsculas, sin acentos, separadores
# (_ - . espacios) colapsados a un espacio, y se le quitan los sufijos de `ruido_sufijos`.
# Luego prueba `exactos` (igualdad) y después `patrones` (regex, re.search).
# El orden importa: gana la primera clase que haga match, en el orden de este archivo.

ruido_sufijos:
  # Estados/atributos que algunos datasets pegan a la especie y no cambian la clase.
  - landing
  - smashed
  - female
  - male
  - hembra
  - macho
  - adult
  - larvae

mapeo:
  aegypti:
    exactos:
      - aegypti
      - aedes aegypti
      - ae aegypti
      - aedes aegypti
    patrones:
      - 'aedes +aegypti'
      - '^ae[ -]*aegypti$'
      - '^aegypti$'

  albopictus:
    exactos:
      - albopictus
      - aedes albopictus
      - ae albopictus
      - aedes albobictus     # error de tipeo real en mosquito-itce6/mosquito_annotation
    patrones:
      - 'aedes +alb[o0]'
      - '^ae[ -]*albopictus$'
      - '^albopictus$'
      - 'tiger mosquito'

  culex:
    exactos:
      - culex
      - cx
    patrones:
      - '^culex'
      - '^cx[ -]'
      - 'quinquefasciatus'
      - 'tritaeniorhynchus'
      - 'vishnui'
      - 'erythrothorax'
      - 'tarsalis'
      - 'pipiens'

  anopheles:
    exactos:
      - anopheles
      - an
    patrones:
      - '^anopheles'
      - '^an[ -]'
      # Especies de Anopheles que algunos datasets rotulan sin el género:
      - 'gambiae'
      - 'funestus'
      - 'stephensi'
      - 'coustani'
      - 'crucians'
      - 'punctipennis'
      - 'quadrimaculatus'
      - 'barbirostris'
      - 'culicifacies'
      - 'subpictus'
      - 'tessellatus'
      - 'culiciformis'
      - 'jamesii'
      - 'vagus'

  otro_mosquito:
    exactos:
      - culiseta
      - armigeres
      - non aedes
      - non_aedes
      - other mosquito
    patrones:
      - 'culiseta'
      - 'japonicus'
      - 'koreicus'
      - 'armigeres'
      - 'vexans'
      - 'vittatus'
      - 'mansonia'
      - 'psorophora'
      # "non aedes"/"no aedes" es un mosquito que no es Aedes — NO es "no es mosquito".
      - '^non[ -]*aedes$'
      - '^no[ -]*aedes$'

  no_es_mosquito:
    exactos:
      - debris
      - background
      - fondo
      - basura
      - not mosquito
      - no mosquito
    patrones:
      - '^debris$'
      - '^background$'
      - '^insect$'         # "Insect" genérico en al-45kdp: insecto no-mosquito
      - '^other insect'
      - '^fly$'
      - '^spider$'
      - '^moth$'

  # --- Ambiguas (detector sí, clasificador no) ---
  aedes_sp:
    exactos:
      - aedes
      - ae
    patrones:
      - '^aedes$'
      - '^aedes sp'

  mosquito_sp:
    exactos:
      - mosquito
      - mosquitos
      - mosquitoes
      - objects
      - object
      - moustique
      - ka
    patrones:
      - '^mosquito'
      - '^mosquitos?$'
      - '^object'

# --- Etiquetas a DESCARTAR por completo -------------------------------------------------------
# Ruido de datasets mal curados: no aportan ni al detector ni al clasificador.
descartar:
  exactos:
    - unlabeled
    - ''
  patrones:
    - '^={3,}'           # "==============================" en dataset-merging/mosquito-cxdah
    - '^\d+$'            # clases numéricas sueltas ("0", "1", "2")
    # Nombre de export que quedó como clase en mosq-nj23o/mosqkito.
    # OJO: los patrones se evalúan contra la etiqueta YA NORMALIZADA (sin guiones ni puntos),
    # así que "- Mosquito Detection - 2023-10-18 12-12am" llega acá como
    # "mosquito detection 2023 10 18 12 12am". Sin la fecha, caería en 'mosquito_sp'.
    - '^mosquito detection \d{4}'

In [ ]:
%%writefile /kaggle/working/config/fuentes.yaml
# Fuentes del dataset consolidado (plan v5, Fase 1). Inventario real verificado el 2026-09-16
# vía búsqueda en Roboflow Universe y Kaggle.
#
# TODAS las licencias listadas son permisivas (CC BY 4.0 / MIT / CDLA-Permissive / dominio
# público). CC BY 4.0 **exige atribución**: ml/dataset/ATRIBUCION.md se genera a partir de este
# archivo y debe acompañar cualquier publicación del dataset o del modelo.
#
# `tier` controla qué se descarga:
#   1 = núcleo, alto valor, se descarga siempre
#   2 = complementario, mejora cobertura/variedad
#   3 = opcional / pesado / a evaluar
#
# `version: null` significa que la búsqueda no devolvió versión publicada: el script prueba la
# última disponible y avisa si falla.

version_inventario: 1
fecha_inventario: '2026-09-16'

roboflow:
  # ---------------------------------------------------------------- Etapa A: detector ---------
  - slug: mosquitos001/mosquito-4ocly
    version: 12
    imagenes: 9895
    tipo: object-detection
    licencia: CC BY 4.0
    clases: [mosquito]
    tier: 1
    uso: [detector]
    nota: >-
      Clase única, el dataset más grande de "hay un mosquito acá". Base del detector.
      77 descargas y un modelo entrenado: señal de que está razonablemente curado.

  - slug: trapmos/mosquito-5wsj2
    version: 1
    imagenes: 10000
    tipo: object-detection
    licencia: CC BY 4.0
    clases: ['Aedes aegypti', 'Aedes albopictus']
    tier: 1
    uso: [detector, clasificador]
    nota: >-
      El nombre del workspace ("trapmos") sugiere imágenes de TRAMPA, que es exactamente la
      distribución de destino del Pi. Prioritario para el test set "tipo campo" (Fase 2).

  # ---------------------------------------------------------------- Etapa B: clasificador -----
  - slug: mosquitos-u6ipx/mosquito-detection-dataset
    version: 4
    imagenes: 7672
    tipo: object-detection
    licencia: CC BY 4.0
    clases: [aegypti, albopictus, anopheles, culex, culiseta, japonicus-koreicus]
    tier: 1
    uso: [detector, clasificador]
    nota: >-
      La mejor cobertura de una sola fuente: cubre las 4 especies objetivo + 2 que caen en
      'otro_mosquito'. Núcleo del clasificador.

  - slug: license-plate-detection-ldjnv/mosquitos-classification
    version: 1
    imagenes: 7239
    tipo: classification
    licencia: CC BY 4.0
    clases: [Ae-aegypti, Ae-albopictus, Ae-vexans, An-tessellatus, Cx-quinquefasciatus, Cx-vishnui, Misc]
    tier: 3   # bajado de 1: ver nota
    estado: 'FALLA 2026-09-16 — la API responde "does not exist or cannot be loaded due to missing permissions" aunque sigue listado en Universe (probablemente sin versión exportable pública). Probar fork vía MCP (projects_fork) si hace falta.'
    uso: [clasificador]
    nota: >-
      Cubre los 3 géneros + 'Misc'. El nombre del workspace no tiene relación con el contenido
      (workspace reutilizado); el dataset sí es de mosquitos.

  - slug: mosquito-tidps/mosquito-detection-yr7y3
    version: 1
    imagenes: 7211
    tipo: classification
    licencia: CC BY 4.0
    clases: ['Aedes aegypti landing', 'Aedes aegypti smashed', 'Aedes albopictus landing',
             'Aedes albopictus smashed', 'Culex landing', 'Culex quinquefasciatus landing',
             'Culex quinquefasciatus smashed', 'Culex smashed', 'aegypti landing',
             'aegypti smashed', 'albopictus landing', 'albopictus smashed']
    tier: 1
    uso: [clasificador]
    nota: >-
      Los sufijos landing/smashed los limpia `ruido_sufijos` de taxonomia.yaml. Ojo: 'smashed'
      (mosquito aplastado) es una distribución distinta a la de campo — útil para variedad,
      pero NO debe dominar el test set.

  - slug: mosquito-itce6/mosquito_annotation
    version: 11
    imagenes: 6800
    tipo: object-detection
    licencia: CC BY 4.0
    clases: ['Aedes aegypti Female', 'Aedes aegypti Male', 'Aedes albopictus Female',
             'Aedes albopictus Male', 'Aedes vittatus Female', 'Aedes vittatus Male',
             'Aedes_aegypti', 'Aedes_albobictus', 'Aedes_vittatus', 'Anopheles culiciformis',
             'Anopheles jamesii', 'Anopheles tessellatus', 'Anopheles_barbirostris',
             'Anopheles_culicifacies', 'Anopheles_stephensi', 'Anopheles_subpictus',
             'Anopheles_vagus', 'Armigeres_subalbatus', 'Culex quinquefasciatus Male',
             'Culex tritaeniorhynchus Female']
    tier: 1
    uso: [detector, clasificador]
    nota: >-
      La fuente MÁS valiosa para 'anopheles': 8 especies distintas del género, que es la clase
      rara en todas las demás. Además trae sexo (ver pregunta abierta §11.3 del plan).

  - slug: mosquitoscan/mosquito-detection-esjag
    version: 2
    imagenes: 4252
    tipo: object-detection
    licencia: CC BY 4.0
    clases: [aegypti, albopictus, culiseta, japonicus-koreicus]
    tier: 1
    uso: [detector, clasificador]

  - slug: angelomontalban629-gmail-com/aedes-species-classifier-clean
    version: 1
    imagenes: 3775
    tipo: classification
    licencia: CC BY 4.0
    clases: ['Aedes Aegypti', 'Aedes Albopictus']
    tier: 1
    uso: [clasificador]
    nota: "El autor lo llama 'clean': verificar en Fase 2 si efectivamente está mejor curado."

  - slug: tickcitizenscience/mosquito-qi01f
    version: 1
    imagenes: 2792
    tipo: object-detection
    licencia: CC BY 4.0
    clases: [aegypti, albopictus, albopictus-aegypti]
    tier: 2
    uso: [detector, clasificador]
    nota: >-
      Ciencia ciudadana => fotos tomadas por gente común con celular. Alto valor para la brecha
      laboratorio↔campo (PLAN_V5 §1.1). La clase 'albopictus-aegypti' es ambigua: se descarta
      para el clasificador y se usa solo como 'mosquito' en el detector.

  - slug: alves-world/dengu-gk6lv
    version: 2
    imagenes: 1367
    tipo: object-detection
    licencia: CC BY 4.0
    clases: ['Aedes aegypti', 'Aedes albopictus', 'Culex quinquefasciatus']
    tier: 2
    uso: [detector, clasificador]

  - slug: mozziev3/mozziev3
    version: 2
    imagenes: 1003
    tipo: object-detection
    licencia: CC BY 4.0
    clases: ['Aedes aegypti Female', 'Aedes aegypti Male', 'Aedes albopictus Female',
             'Aedes albopictus Male', 'Non Aedes']
    tier: 2
    uso: [detector, clasificador]
    nota: "'Non Aedes' → otro_mosquito (es mosquito, no es Aedes). No confundir con no_es_mosquito."

  - slug: denguemetric-o9fa8/aedes-classification
    version: 2
    imagenes: 931
    tipo: object-detection
    licencia: CC BY 4.0
    clases: ['aedes aegypti ', 'aedes albopictus']
    tier: 2
    uso: [detector, clasificador]

  - slug: jhon-mark-blay/anopheles
    version: 1
    imagenes: 957
    tipo: classification
    licencia: Public Domain
    clases: [coustani, crucians, funestus, gambiae, punctipennis, quadrimaculatus]
    tier: 1
    uso: [clasificador]
    nota: >-
      6 especies de Anopheles sin el género en la etiqueta: taxonomia.yaml las captura por
      patrón. Segunda fuente clave para la clase rara.

  - slug: ks-job/anopheles-nmkvs
    version: 1
    imagenes: 215
    tipo: instance-segmentation
    licencia: CC BY 4.0
    clases: [anopheles]
    tier: 2
    uso: [detector, clasificador]
    nota: "Segmentación: el script usa solo la caja envolvente de cada máscara."

  - slug: sebskie-uvuoi/mosquito-6vtkg
    version: 1
    imagenes: 1400
    tipo: object-detection
    licencia: CC BY 4.0
    clases: ['Aedes Aegypti', 'Aedes Albopictus']
    tier: 2
    uso: [detector, clasificador]
    nota: >-
      ⚠️ Mismo workspace que val-ust52: probablemente sean el mismo material partido en dos.
      La deduplicación por pHash de la Fase 2 debe resolverlo — verificar el reporte.

  - slug: sebskie-uvuoi/val-ust52
    version: 2
    imagenes: 599
    tipo: object-detection
    licencia: CC BY 4.0
    clases: ['Aedes Aegypti', 'Aedes Albopictus']
    tier: 3
    uso: [detector, clasificador]
    nota: "⚠️ Ver nota de mosquito-6vtkg. El nombre 'val' sugiere que es el split de validación."

  - slug: mosq-nj23o/mosqkito
    version: 1
    imagenes: 1000
    tipo: object-detection
    licencia: CC BY 4.0
    clases: [AEDES, 'Aedes Aegypti', Mosquito, aegypti, albopictus, anopheles, culex,
             culiseta, japonicus-koreicus, mosquito]
    tier: 3
    uso: [detector, clasificador]
    nota: >-
      Mezcla etiquetas de varias fuentes (incluye 'AEDES' ambiguo). Alto riesgo de solapamiento
      con otros datasets de esta lista: usar solo si la dedup lo deja limpio.

  - slug: al-45kdp/mosquito-species
    version: null
    imagenes: 78
    tipo: object-detection
    licencia: CC BY 4.0
    clases: ['Culex erythrothorax', 'Culex tarsalis', 'Culiseta inornata', Debris, Insect, Mosquito]
    tier: 3
    uso: [detector, clasificador]
    nota: >-
      Chico, pero es de las pocas fuentes con 'Debris' e 'Insect' — negativos reales de trampa,
      justo lo que necesita la clase no_es_mosquito.

  # ---------------------------------------------------------------- Negativos duros -----------
  # `clase_por_defecto: no_es_mosquito` → toda etiqueta que NO resuelva a mosquito en
  # taxonomia.yaml cae en no_es_mosquito. Si alguna sí resuelve (ej. la clase 'Mosquito' de un
  # dataset de insectos), manda la taxonomía. Probado en test_taxonomia.py.
  # Agregados 2026-09-16: la corrida v3 dejó no_es_mosquito con 0 imágenes.

  - slug: maximilian-sittinger/insect_detect_classification
    version: 2
    imagenes: 2422
    tipo: classification
    licencia: CC BY 4.0
    clases: [episyr_balt, fly, hbee, hovfly, other, shadow, wasp]
    clase_por_defecto: no_es_mosquito
    tier: 1
    uso: [clasificador]
    nota: >-
      El negativo MÁS valioso: sale de "Insect Detect", una cámara trampa DIY de insectos. Las
      clases 'shadow' y 'other' son exactamente los falsos positivos que va a ver una trampa.
      Ojo: la v2 del mismo autor es BY-NC-SA — no usar esa.

  - slug: comsats-lbpbe/insects-2fw0a
    version: 2
    imagenes: 1917
    tipo: classification
    licencia: CC BY 4.0
    clases: ['Africanized Honey Bees (Killer Bees)', Aphids, Armyworms, 'Fruit Flies', Grasshopper,
             Mosquito, Sawfly, 'Spider Mites', Thrips, '... (18 en total)']
    clase_por_defecto: no_es_mosquito
    tier: 1
    uso: [clasificador]
    nota: "Trae una clase 'Mosquito': resuelve a mosquito_sp y queda FUERA de los negativos."

  - slug: neung-s7rrh/moth-juac0
    version: 2
    imagenes: 7702
    tipo: classification
    licencia: CC BY 4.0
    clases: ['50 especies de polillas']
    clase_por_defecto: no_es_mosquito
    max_imagenes: 1200
    tier: 1
    uso: [clasificador]
    nota: >-
      Las polillas chicas se confunden con mosquitos en trampas nocturnas. Tope de 1 200 para que
      no dominen la clase frente a los negativos de cámara trampa.

  - slug: mosquito1/americano-y8n5a
    version: 2
    imagenes: 14640
    tipo: classification
    licencia: CC BY 4.0
    clases: [aedes, non_aedes]
    max_imagenes: 1500   # sin tope, 'non_aedes' triplicaría otro_mosquito (2 235 en la corrida v3)
    tier: 2
    uso: [clasificador]
    nota: >-
      El dataset más grande del inventario. 'aedes' es ambiguo (no dice especie) => se descarta
      para el clasificador; el valor está en 'non_aedes' como otro_mosquito. Revisar una muestra
      antes de confiar: si 'non_aedes' incluye insectos no-culícidos, parte va a no_es_mosquito.

kaggle:
  - slug: ahsanatiq/mosquitos-classification-images
    imagenes: 3000
    licencia: MIT
    requiere_input: true   # corridas no interactivas: agregar en el editor (Add Input → Datasets)
    clases: [AEDES, ANOPHELES, CULEX]
    tier: 1
    uso: [clasificador]
    nota: >-
      Etiquetas a nivel GÉNERO. 'ANOPHELES' y 'CULEX' se usan directo (son clases canónicas);
      'AEDES' es ambiguo => aedes_sp, solo detector. Licencia MIT, la más permisiva del lote.

  - slug: cyberthorn/chula-mosquito-classification
    imagenes: null
    licencia: CDLA-Permissive-1.0
    clases: ['6 especies de Tailandia + misceláneo']
    tier: 3
    uso: [clasificador]
    nota: >-
      5,5 GB de microscopía. Muy limpio pero MUY lejos de la distribución de campo: si domina el
      entrenamiento, empeora el rendimiento real. Evaluar recién en Fase 4 y con peso bajo.

# ---------------------------------------------------------------- Pendiente de buscar ---------
faltantes:
  - que: Dataset de insectos genéricos (moscas, polillas, arañas) para 'no_es_mosquito'
    por_que: >-
      Hoy los negativos salen solo de 'Debris'/'Insect' de al-45kdp (78 imágenes) y quizá de
      'non_aedes'. Es poco: el modelo va a rechazar mal las fotos de otros bichos, que en campo
      van a ser frecuentes.
    donde_buscar: [Roboflow Universe 'insects', Kaggle 'insect classification', iNaturalist]

  - que: Fotos propias de trampas del proyecto
    por_que: >-
      Definirían el test set "tipo campo" real. Confirmado con el usuario (2026-09-16) que
      todavía no existen; hasta que existan, el sustituto es trapmos + tickcitizenscience.
    estado: no disponible

In [ ]:
%%writefile /kaggle/working/consolidar.py
"""
Fase 2 del plan v5 — Consolidación del dataset de mosquitos.

Corre en un notebook de Kaggle. Descarga todas las fuentes de ml/dataset/fuentes.yaml
(Roboflow Universe + Kaggle), normaliza las etiquetas con ml/dataset/taxonomia.yaml,
deduplica, y produce DOS datasets listos para entrenar:

    detector/       YOLO, una sola clase 'mosquito'          → Fase 3
    clasificador/   carpetas por clase, recortes de mosquito → Fase 4

Nada de esto toca el disco de la máquina del usuario: se descarga dentro de Kaggle y se
publica como Kaggle Dataset.

--------------------------------------------------------------------------------------
REQUISITOS EN EL NOTEBOOK DE KAGGLE
--------------------------------------------------------------------------------------
1. Settings → Persistence → "Files only" (para que /kaggle/working sobreviva).
2. Add-ons → Secrets → agregar `ROBOFLOW_API_KEY` (clave privada de Roboflow).
   NUNCA pegar la clave en el código ni en el chat.
3. Internet: ON.
4. Acelerador: no hace falta GPU en esta fase (es I/O y CPU).

    !pip -q install roboflow imagehash pyyaml pillow tqdm

--------------------------------------------------------------------------------------
DECISIÓN DE DISEÑO IMPORTANTE — por qué se agrupa por pHash en vez de solo borrar duplicados
--------------------------------------------------------------------------------------
Varias fuentes de Universe son re-subidas del mismo material original. Si la misma imagen
cae en train y en test, las métricas salen infladas y el modelo parece excelente hasta que
llega al campo.

Borrar duplicados "perfectamente" es frágil (recompresiones, reescalados, recortes leves).
Entonces además de borrar los exactos, el split se hace **por grupo de pHash**: aunque
sobreviva un duplicado, todas sus copias caen en el MISMO split. Eso elimina la fuga de
datos aunque la deduplicación no sea perfecta.
"""

from __future__ import annotations

import json
import os
import random
import re
import shutil
import unicodedata
from collections import Counter, defaultdict
from dataclasses import dataclass, field, asdict
from pathlib import Path

import yaml
from PIL import Image
from tqdm.auto import tqdm

# --------------------------------------------------------------------------------------
# Configuración
# --------------------------------------------------------------------------------------

SALIDA = Path("/kaggle/working/mosquito-merged-v1")
DESCARGAS = Path("/kaggle/temp/descargas")   # /kaggle/temp no cuenta para el output
VERSION_DATASET = "v1"

# Qué tiers de fuentes.yaml descargar. Empezar con [1] para una corrida rápida.
TIERS = [1, 2]

SPLIT = {"train": 0.70, "valid": 0.15, "test": 0.15}
SEMILLA = 1312

# Margen alrededor de la caja al recortar para el clasificador. El contexto (patas, postura)
# ayuda a distinguir Anopheles, que se posa inclinado.
MARGEN_RECORTE = 0.15
LADO_MINIMO_RECORTE = 32   # recortes más chicos que esto no tienen detalle de tórax utilizable

random.seed(SEMILLA)

# --------------------------------------------------------------------------------------
# Taxonomía
# --------------------------------------------------------------------------------------


def normalizar(etiqueta: str) -> str:
    """minúsculas, sin acentos, separadores colapsados. Debe coincidir con lo que documenta
    taxonomia.yaml, porque los patrones del YAML se escriben contra ESTA forma."""
    s = unicodedata.normalize("NFKD", str(etiqueta))
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.lower()
    s = re.sub(r"[_\-.]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


class Taxonomia:
    def __init__(self, ruta: Path):
        self.cfg = yaml.safe_load(ruta.read_text(encoding="utf-8"))
        self.canonicas = list(self.cfg["clases_canonicas"])
        self.ambiguas = list(self.cfg["ambiguas"])
        self.det = self.cfg["detector"]
        self.ruido = [normalizar(s) for s in self.cfg.get("ruido_sufijos", [])]
        self._descartar = self.cfg.get("descartar", {})
        self._mapeo = self.cfg["mapeo"]
        self._cache: dict[str, str | None] = {}

    def _quitar_ruido(self, s: str) -> str:
        cambio = True
        while cambio:
            cambio = False
            for suf in self.ruido:
                if s.endswith(" " + suf):
                    s = s[: -(len(suf) + 1)].strip()
                    cambio = True
                if s.startswith(suf + " "):
                    s = s[len(suf) + 1 :].strip()
                    cambio = True
        return s

    def es_descartable(self, etiqueta: str) -> bool:
        """Etiquetas basura explícitas (Unlabeled, '====', clases numéricas). A diferencia de una
        etiqueta simplemente no reconocida, estas NUNCA reciben la clase por defecto de la fuente."""
        s = self._quitar_ruido(normalizar(etiqueta))
        if not s or s in [normalizar(x) for x in self._descartar.get("exactos", [])]:
            return True
        return any(re.search(p, s) for p in self._descartar.get("patrones", []))

    def resolver_con_defecto(self, etiqueta: str, defecto: str | None) -> str | None:
        """Como `resolver`, pero en fuentes de negativos (moscas, polillas...) las etiquetas que no
        son de mosquito caen en `defecto` ('no_es_mosquito'). Si la etiqueta SÍ resuelve a una clase
        de mosquito (ej. la clase 'Mosquito' de un dataset de insectos), manda la taxonomía."""
        clase = self.resolver(etiqueta)
        if clase is None and defecto and not self.es_descartable(etiqueta):
            return defecto
        return clase

    def resolver(self, etiqueta: str) -> str | None:
        """Devuelve la clase canónica (o ambigua), o None si hay que descartar la etiqueta."""
        if etiqueta in self._cache:
            return self._cache[etiqueta]
        s = self._quitar_ruido(normalizar(etiqueta))
        resultado = None

        if s in [normalizar(x) for x in self._descartar.get("exactos", [])] or not s:
            resultado = None
        elif any(re.search(p, s) for p in self._descartar.get("patrones", [])):
            resultado = None
        else:
            for clase, reglas in self._mapeo.items():
                exactos = [normalizar(x) for x in reglas.get("exactos", [])]
                if s in exactos:
                    resultado = clase
                    break
                if any(re.search(p, s) for p in reglas.get("patrones", [])):
                    resultado = clase
                    break

        self._cache[etiqueta] = resultado
        return resultado

    def sirve_para_clasificador(self, clase: str | None) -> bool:
        # Las ambiguas ('aedes_sp', 'mosquito_sp') son mosquitos, pero no dicen especie:
        # entrenar el clasificador con ellas sería enseñarle ruido.
        return clase is not None and clase in self.canonicas

    def sirve_para_detector(self, clase: str | None) -> bool:
        return clase is not None and clase in self.det["incluye"]


# --------------------------------------------------------------------------------------
# Registro de imágenes
# --------------------------------------------------------------------------------------


@dataclass
class Muestra:
    """Una imagen con su procedencia. `phash` es la clave de agrupación para el split."""
    ruta: str
    fuente: str
    etiqueta_original: str
    clase: str                       # canónica o ambigua
    cajas: list = field(default_factory=list)   # [(clase, cx, cy, w, h)] normalizadas YOLO
    phash: str | None = None
    split: str | None = None


def cargar_config(raiz: Path) -> tuple[Taxonomia, dict]:
    tax = Taxonomia(raiz / "taxonomia.yaml")
    fuentes = yaml.safe_load((raiz / "fuentes.yaml").read_text(encoding="utf-8"))
    return tax, fuentes


# --------------------------------------------------------------------------------------
# Descarga
# --------------------------------------------------------------------------------------


def descargar_roboflow(fuentes: list[dict], destino: Path) -> dict[str, Path]:
    """Descarga cada proyecto público de Universe. Si uno falla, sigue con el resto:
    una fuente caída no debe tumbar una corrida de horas."""
    from roboflow import Roboflow
    from kaggle_secrets import UserSecretsClient

    api_key = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
    rf = Roboflow(api_key=api_key)

    formato = {
        "object-detection": "yolov8",
        "instance-segmentation": "yolov8",
        "classification": "folder",
    }

    rutas: dict[str, Path] = {}
    for f in tqdm(fuentes, desc="Roboflow"):
        slug = f["slug"]
        ws, proj = slug.split("/")
        carpeta = destino / ws / proj
        if carpeta.exists() and any(carpeta.iterdir()):
            rutas[slug] = carpeta
            continue
        try:
            p = rf.workspace(ws).project(proj)
            version = f.get("version")
            if not version:
                # `Version.version` puede venir como "workspace/project/3": nos quedamos con el
                # último segmento numérico en vez de asumir que ya es un entero.
                disponibles = []
                for v in p.versions():
                    try:
                        disponibles.append(int(str(v.version).rstrip("/").split("/")[-1]))
                    except (ValueError, AttributeError):
                        continue
                if not disponibles:
                    raise RuntimeError("no se pudo determinar la versión publicada")
                version = max(disponibles)
                print(f"  {slug}: sin versión fijada, uso la {version}")
            p.version(int(version)).download(
                formato[f["tipo"]], location=str(carpeta), overwrite=False
            )
            rutas[slug] = carpeta
        except Exception as e:                                    # noqa: BLE001
            print(f"  ⚠️  {slug}: {type(e).__name__}: {e}")
    return rutas


def descargar_kaggle(fuentes: list[dict]) -> dict[str, Path]:
    """En una corrida no interactiva (Save & Run All) Kaggle NO deja adjuntar datasets nuevos con
    kagglehub ('New Datasets cannot be attached in non-interactive sessions'): hay que agregarlos
    como Input en el editor antes de lanzar. Por eso se busca primero en /kaggle/input."""
    import kagglehub

    rutas: dict[str, Path] = {}
    for f in tqdm(fuentes, desc="Kaggle"):
        nombre = f["slug"].split("/")[-1]
        montados = [p for p in Path("/kaggle/input").rglob(nombre) if p.is_dir()]
        if montados:
            rutas[f["slug"]] = montados[0]
            continue
        try:
            rutas[f["slug"]] = Path(kagglehub.dataset_download(f["slug"]))
        except Exception as e:                                    # noqa: BLE001
            print(f"  ⚠️  {f['slug']}: {type(e).__name__}: {e}")
            print(f"      → Agregalo en el editor: Add Input → Datasets → {f['slug']}")
    return rutas


# --------------------------------------------------------------------------------------
# Lectura de cada formato
# --------------------------------------------------------------------------------------

EXT_IMG = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def leer_yolo(raiz: Path, slug: str, tax: Taxonomia, defecto: str | None = None) -> list[Muestra]:
    """Formato YOLO de Roboflow: data.yaml + {split}/images + {split}/labels.
    Los splits originales se ignoran: se rearman desde cero para que el split sea consistente
    entre todas las fuentes y respete los grupos de pHash."""
    data_yaml = next(raiz.rglob("data.yaml"), None)
    if not data_yaml:
        return []
    nombres = yaml.safe_load(data_yaml.read_text(encoding="utf-8")).get("names", [])
    if isinstance(nombres, dict):
        nombres = [nombres[k] for k in sorted(nombres)]

    muestras: list[Muestra] = []
    for img in raiz.rglob("*"):
        if img.suffix.lower() not in EXT_IMG or "labels" in img.parts:
            continue
        lbl = Path(str(img).replace("/images/", "/labels/")).with_suffix(".txt")
        if not lbl.exists():
            continue

        cajas, originales = [], []
        for linea in lbl.read_text(encoding="utf-8").strip().splitlines():
            partes = linea.split()
            if len(partes) < 5:
                continue
            idx = int(float(partes[0]))
            if idx >= len(nombres):
                continue
            original = nombres[idx]
            clase = tax.resolver_con_defecto(original, defecto)
            if clase is None:
                continue
            coords = [float(x) for x in partes[1:]]
            # Segmentación: el polígono se reduce a su caja envolvente.
            if len(coords) > 4:
                xs, ys = coords[0::2], coords[1::2]
                cx, cy = (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2
                w, h = max(xs) - min(xs), max(ys) - min(ys)
            else:
                cx, cy, w, h = coords[:4]
            cajas.append((clase, cx, cy, w, h))
            originales.append(original)

        if not cajas:
            continue
        dominante = Counter(c[0] for c in cajas).most_common(1)[0][0]
        muestras.append(
            Muestra(
                ruta=str(img),
                fuente=slug,
                etiqueta_original=";".join(sorted(set(originales))),
                clase=dominante,
                cajas=cajas,
            )
        )
    return muestras


def leer_carpetas(raiz: Path, slug: str, tax: Taxonomia, defecto: str | None = None) -> list[Muestra]:
    """Formato 'folder' (clasificación): una carpeta por clase. Sirve tanto para los exports
    de Roboflow como para los datasets de Kaggle organizados así."""
    muestras: list[Muestra] = []
    for img in raiz.rglob("*"):
        if img.suffix.lower() not in EXT_IMG:
            continue
        original = img.parent.name
        clase = tax.resolver_con_defecto(original, defecto)
        if clase is None:
            continue
        muestras.append(
            Muestra(ruta=str(img), fuente=slug, etiqueta_original=original, clase=clase)
        )
    return muestras


def leer_fuente(ruta: Path, f: dict, tax: Taxonomia) -> list[Muestra]:
    defecto = f.get("clase_por_defecto")
    if f.get("tipo") == "classification" or next(ruta.rglob("data.yaml"), None) is None:
        muestras = leer_carpetas(ruta, f["slug"], tax, defecto)
    else:
        muestras = leer_yolo(ruta, f["slug"], tax, defecto)
    tope = f.get("max_imagenes")
    if tope and len(muestras) > tope:
        # Tope por fuente: evita que una fuente grande de negativos (7 700 polillas) domine su
        # clase. Muestreo con semilla fija para que la consolidación sea reproducible.
        muestras = random.Random(SEMILLA).sample(muestras, tope)
    return muestras


# --------------------------------------------------------------------------------------
# Deduplicación y split
# --------------------------------------------------------------------------------------


def calcular_phash(muestras: list[Muestra]) -> None:
    import imagehash

    for m in tqdm(muestras, desc="pHash"):
        try:
            with Image.open(m.ruta) as im:
                m.phash = str(imagehash.phash(im.convert("RGB")))
        except Exception:                                          # noqa: BLE001
            m.phash = None   # imagen corrupta: se descarta más abajo


def deduplicar(muestras: list[Muestra]) -> tuple[list[Muestra], dict]:
    """Quita imágenes corruptas y copias exactas (mismo pHash). Conserva la primera aparición,
    priorizando las fuentes con cajas (sirven para ambas etapas)."""
    validas = [m for m in muestras if m.phash]
    corruptas = len(muestras) - len(validas)

    validas.sort(key=lambda m: (0 if m.cajas else 1, m.fuente))
    vistos: set[str] = set()
    unicas: list[Muestra] = []
    for m in validas:
        if m.phash in vistos:
            continue
        vistos.add(m.phash)
        unicas.append(m)

    return unicas, {
        "entrada": len(muestras),
        "corruptas": corruptas,
        "duplicadas_exactas": len(validas) - len(unicas),
        "salida": len(unicas),
    }


def stem_original(ruta: str) -> str:
    """Nombre de la imagen ORIGINAL de la que salió una exportación de Roboflow.

    Roboflow exporta cada variante augmentada como `<original>_<ext>.rf.<hash>.<ext>`
    (ej. `IMG_123_jpg.rf.9f2c...jpg`). Todas las variantes de una misma foto comparten el prefijo
    antes de `.rf.`. Hace falta porque una imagen espejada o rotada tiene OTRO pHash: sin esta
    clave, las augmentaciones de una misma foto podían caer una en train y otra en test (fuga de
    datos detectada en la corrida v3 de la consolidación, 2026-09-16)."""
    nombre = Path(ruta).name
    base = nombre.split(".rf.")[0] if ".rf." in nombre else Path(nombre).stem
    return re.sub(r"_(jpe?g|png|bmp|webp)$", "", base, flags=re.IGNORECASE)


def agrupar(muestras: list[Muestra]) -> list[list[Muestra]]:
    """Une en un mismo grupo las muestras que comparten pHash O imagen original (union-find).
    Todo el grupo va al mismo split."""
    padre = list(range(len(muestras)))

    def raiz(i: int) -> int:
        while padre[i] != i:
            padre[i] = padre[padre[i]]
            i = padre[i]
        return i

    def unir(a: int, b: int) -> None:
        ra, rb = raiz(a), raiz(b)
        if ra != rb:
            padre[rb] = ra

    primero_por_clave: dict[str, int] = {}
    for i, m in enumerate(muestras):
        for clave in (f"ph:{m.phash}", f"st:{m.fuente}/{stem_original(m.ruta)}"):
            if clave in primero_por_clave:
                unir(i, primero_por_clave[clave])
            else:
                primero_por_clave[clave] = i

    grupos: dict[int, list[Muestra]] = defaultdict(list)
    for i, m in enumerate(muestras):
        grupos[raiz(i)].append(m)
    return list(grupos.values())


def asignar_split(muestras: list[Muestra]) -> tuple[list[Muestra], dict]:
    """Split estratificado por clase y AGRUPADO (ver `agrupar`).

    En valid y test se conserva UNA sola muestra por grupo: las variantes augmentadas que ya traía
    la exportación de Roboflow sirven para entrenar, pero en evaluación inflarían el conteo con
    copias de la misma foto y el test dejaría de ser un conjunto limpio."""
    grupos = agrupar(muestras)

    por_clase: dict[str, list[list[Muestra]]] = defaultdict(list)
    for g in grupos:
        clase = Counter(m.clase for m in g).most_common(1)[0][0]
        por_clase[clase].append(g)

    conservadas: list[Muestra] = []
    descartadas_eval = 0
    conteo = {s: Counter() for s in SPLIT}
    for clase, gs in sorted(por_clase.items()):
        random.shuffle(gs)
        n = len(gs)
        n_train = int(n * SPLIT["train"])
        n_valid = int(n * SPLIT["valid"])
        tramos = {
            "train": gs[:n_train],
            "valid": gs[n_train : n_train + n_valid],
            "test": gs[n_train + n_valid :],
        }
        for split, grupos_split in tramos.items():
            for g in grupos_split:
                if split != "train" and len(g) > 1:
                    g = sorted(g, key=lambda m: m.ruta)[:1]   # determinista
                    descartadas_eval += 1
                for m in g:
                    m.split = split
                    conservadas.append(m)
                conteo[split][clase] += len(g)

    return conservadas, {
        "grupos": len(grupos),
        "grupos_con_variantes": sum(1 for g in grupos if len(g) > 1),
        "variantes_quitadas_de_valid_test": descartadas_eval,
        "por_split": {s: dict(c) for s, c in conteo.items()},
    }


# --------------------------------------------------------------------------------------
# Escritura de los dos datasets
# --------------------------------------------------------------------------------------


def escribir_detector(muestras: list[Muestra], tax: Taxonomia, salida: Path) -> dict:
    """YOLO con una sola clase. Solo usa muestras con cajas (las de clasificación no tienen
    coordenadas, así que no sirven para enseñar a localizar)."""
    base = salida / "detector"
    for split in SPLIT:
        (base / split / "images").mkdir(parents=True, exist_ok=True)
        (base / split / "labels").mkdir(parents=True, exist_ok=True)

    conteo = Counter()
    for m in tqdm([m for m in muestras if m.cajas], desc="detector"):
        cajas = [c for c in m.cajas if tax.sirve_para_detector(c[0])]
        if not cajas:
            continue
        destino = base / m.split
        nombre = f"{m.phash}{Path(m.ruta).suffix.lower()}"
        shutil.copy2(m.ruta, destino / "images" / nombre)
        (destino / "labels" / f"{m.phash}.txt").write_text(
            "\n".join(f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}" for _, cx, cy, w, h in cajas),
            encoding="utf-8",
        )
        conteo[m.split] += 1

    (base / "data.yaml").write_text(
        yaml.safe_dump(
            {
                "path": str(base),
                "train": "train/images",
                "val": "valid/images",
                "test": "test/images",
                "nc": 1,
                "names": [tax.det["clase_unica"]],
            },
            sort_keys=False,
            allow_unicode=True,
        ),
        encoding="utf-8",
    )
    return dict(conteo)


def escribir_clasificador(muestras: list[Muestra], tax: Taxonomia, salida: Path) -> dict:
    """Carpetas por clase con RECORTES. Las fuentes con cajas aportan un recorte por caja;
    las de clasificación aportan la imagen completa (ya vienen encuadradas)."""
    base = salida / "clasificador"
    for split in SPLIT:
        for clase in tax.canonicas:
            (base / split / clase).mkdir(parents=True, exist_ok=True)

    conteo: dict[str, Counter] = {s: Counter() for s in SPLIT}
    descartados_chicos = 0

    for m in tqdm(muestras, desc="clasificador"):
        try:
            with Image.open(m.ruta) as im:
                im = im.convert("RGB")
                W, H = im.size

                if not m.cajas:
                    if not tax.sirve_para_clasificador(m.clase):
                        continue
                    destino = base / m.split / m.clase / f"{m.phash}.jpg"
                    im.save(destino, "JPEG", quality=92)
                    conteo[m.split][m.clase] += 1
                    continue

                for i, (clase, cx, cy, w, h) in enumerate(m.cajas):
                    if not tax.sirve_para_clasificador(clase):
                        continue
                    bw, bh = w * W * (1 + MARGEN_RECORTE), h * H * (1 + MARGEN_RECORTE)
                    x0 = max(0, int(cx * W - bw / 2))
                    y0 = max(0, int(cy * H - bh / 2))
                    x1 = min(W, int(cx * W + bw / 2))
                    y1 = min(H, int(cy * H + bh / 2))
                    if (x1 - x0) < LADO_MINIMO_RECORTE or (y1 - y0) < LADO_MINIMO_RECORTE:
                        descartados_chicos += 1
                        continue
                    destino = base / m.split / clase / f"{m.phash}_{i}.jpg"
                    im.crop((x0, y0, x1, y1)).save(destino, "JPEG", quality=92)
                    conteo[m.split][clase] += 1
        except Exception:                                          # noqa: BLE001
            continue

    return {
        "por_split": {s: dict(c) for s, c in conteo.items()},
        "descartados_por_tamano": descartados_chicos,
    }


# --------------------------------------------------------------------------------------
# Reporte
# --------------------------------------------------------------------------------------


def escribir_reporte(salida: Path, info: dict) -> None:
    (salida / "MANIFIESTO.json").write_text(
        json.dumps(info, indent=2, ensure_ascii=False), encoding="utf-8"
    )

    cls = info["clasificador"]["por_split"]
    clases = sorted({c for s in cls.values() for c in s})
    filas = ["| clase | train | valid | test | total |", "|---|---|---|---|---|"]
    for c in clases:
        tr, va, te = cls["train"].get(c, 0), cls["valid"].get(c, 0), cls["test"].get(c, 0)
        filas.append(f"| {c} | {tr} | {va} | {te} | {tr + va + te} |")

    dedup = info["dedup"]
    md = f"""# Dataset consolidado de mosquitos — {VERSION_DATASET}

Generado por `ml/notebooks/01_consolidar_dataset.py` (plan v5, Fase 2).
Tiers incluidos: {TIERS} · semilla: {SEMILLA}

## Deduplicación
- Imágenes leídas: **{dedup['entrada']}**
- Corruptas descartadas: {dedup['corruptas']}
- Duplicados exactos (mismo pHash) eliminados: **{dedup['duplicadas_exactas']}**
- Imágenes únicas: **{dedup['salida']}**

> El split se hace por grupo de pHash, así que ninguna copia de una misma imagen puede caer
> en dos splits distintos. Es lo que evita inflar las métricas.

## Clasificador (recortes por clase)
{chr(10).join(filas)}

Recortes descartados por ser menores a {LADO_MINIMO_RECORTE} px: {info['clasificador']['descartados_por_tamano']}

## Detector (una clase)
{json.dumps(info['detector'], indent=2, ensure_ascii=False)}

## Fuentes efectivamente usadas
{chr(10).join(f'- `{k}`: {v} imágenes' for k, v in sorted(info['por_fuente'].items()))}

## Criterios de aceptación de la Fase 2
- [{'x' if dedup['salida'] >= 25000 else ' '}] ≥ 25 000 imágenes utilizables
- [{'x' if all(sum(cls[s].get(c, 0) for s in cls) >= 1500 for c in ['aegypti', 'albopictus', 'culex', 'anopheles']) else ' '}] ≥ 1 500 por clase objetivo
- [{'x' if sum(cls[s].get('no_es_mosquito', 0) for s in cls) >= 1000 else ' '}] ≥ 1 000 negativos (`no_es_mosquito`)
- [x] 0 duplicados ni variantes augmentadas cruzando splits (garantizado por diseño)

## Agrupación anti-fuga
- Grupos: {info['splits']['grupos']}
- Grupos con variantes augmentadas de la exportación de Roboflow: {info['splits']['grupos_con_variantes']}
- Variantes quitadas de valid/test (en evaluación queda 1 por foto original): {info['splits']['variantes_quitadas_de_valid_test']}
"""
    (salida / "REPORTE.md").write_text(md, encoding="utf-8")
    print(md)


# --------------------------------------------------------------------------------------
# Main
# --------------------------------------------------------------------------------------


def main(raiz_config: Path) -> None:
    tax, fuentes = cargar_config(raiz_config)
    SALIDA.mkdir(parents=True, exist_ok=True)
    DESCARGAS.mkdir(parents=True, exist_ok=True)

    rf_fuentes = [f for f in fuentes.get("roboflow", []) if f.get("tier") in TIERS]
    kg_fuentes = [f for f in fuentes.get("kaggle", []) if f.get("tier") in TIERS]
    print(f"Fuentes seleccionadas: {len(rf_fuentes)} Roboflow + {len(kg_fuentes)} Kaggle\n")

    rutas = descargar_roboflow(rf_fuentes, DESCARGAS)
    rutas.update(descargar_kaggle(kg_fuentes))

    muestras: list[Muestra] = []
    por_fuente: dict[str, int] = {}
    for f in rf_fuentes + kg_fuentes:
        ruta = rutas.get(f["slug"])
        if not ruta:
            continue
        leidas = leer_fuente(ruta, f, tax)
        por_fuente[f["slug"]] = len(leidas)
        muestras.extend(leidas)
        print(f"  {f['slug']}: {len(leidas)} imágenes utilizables")

    print(f"\nTotal leído: {len(muestras)}")
    calcular_phash(muestras)
    muestras, dedup = deduplicar(muestras)
    print(f"Tras deduplicar: {dedup['salida']} (-{dedup['duplicadas_exactas']} duplicados)")

    muestras, splits = asignar_split(muestras)
    print(f"Grupos: {splits['grupos']} ({splits['grupos_con_variantes']} con variantes augmentadas)")
    info = {
        "version": VERSION_DATASET,
        "tiers": TIERS,
        "semilla": SEMILLA,
        "dedup": dedup,
        "splits": splits,
        "por_fuente": por_fuente,
        "detector": escribir_detector(muestras, tax, SALIDA),
        "clasificador": escribir_clasificador(muestras, tax, SALIDA),
    }
    escribir_reporte(SALIDA, info)


if __name__ == "__main__":
    # En Kaggle: subir taxonomia.yaml y fuentes.yaml como Dataset de entrada, o clonar el repo.
    raiz = Path(os.environ.get("CONFIG_DIR", "/kaggle/input/mosquito-config"))
    main(raiz)

## Corrida

Corrida completa `TIERS = [1, 2]`. La v3 (solo tier 1) ya validó el pipeline en 23 min;
esta debería tardar ~40-60 min.

In [ ]:
import sys, importlib
from pathlib import Path

sys.path.insert(0, "/kaggle/working")
import consolidar
importlib.reload(consolidar)

TIERS = [1, 2]         # corrida completa (la v3 con [1] validó el pipeline)

consolidar.TIERS = TIERS
consolidar.main(Path("/kaggle/working/config"))

## Resultado

Revisar en el reporte:
- ¿≥ 25 000 imágenes únicas?
- ¿≥ 1 500 por clase objetivo? **`anopheles` es la que más riesgo tiene de quedar corta.**
- ¿Cuántos duplicados eliminó? Si son muchísimos, varias fuentes eran el mismo material.

Si todo está bien: *Save Version* y publicar `/kaggle/working/mosquito-merged-v1` como Dataset
`mosquito-merged-v1` (entrada de las Fases 3 y 4).

In [ ]:
!du -sh /kaggle/working/mosquito-merged-v1/* 2>/dev/null
!find /kaggle/working/mosquito-merged-v1/clasificador -type f | awk -F/ '{print $(NF-2)"/"$(NF-1)}' | sort | uniq -c